# Phase 2 — Dependency Ground-Truth Validation (Final Reproducibility Audit)

**Scope:** Reproduce the processed dependency history → eligible ADD events → 14-day checkpoint labels. This validates *pipeline consistency*, **not** independent truth of upstream Jira extraction, event direction, migration flags, or risk construct validity. Inputs are read-only; outputs are written to a separate evaluation folder.

Expected inputs: `final_dependency_history.csv`, `final_eligible_dependency_adds.csv`, `dependency_escalation_labels.csv`. The notebook discovers unique files in Google Drive, or allows explicit paths / manual uploads.

In [7]:
import pandas as pd, numpy as np, json, hashlib
from pathlib import Path
from datetime import timedelta
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive')
except ImportError:
    DRIVE = Path('/mnt/data')
OUT = DRIVE/'02_Risk_Prediction'/'Label_Validation'/'Results'/'Dependency_Final'
OUT.mkdir(parents=True, exist_ok=True)
print('Results:', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results: /content/drive/MyDrive/02_Risk_Prediction/Label_Validation/Results/Dependency_Final


## 1. Input file resolution
Input paths must resolve to unambiguous source files. The three required CSV inputs must be available in the execution environment.

In [8]:
from pathlib import Path

INPUT_DIR = Path(
    '/content/drive/MyDrive/02_Risk_Prediction/'
    'Label_Validation/Input_Data'
)

HISTORY_PATH = INPUT_DIR / 'final_dependency_history.csv'
ELIGIBLE_PATH = INPUT_DIR / 'final_eligible_dependency_adds.csv'
LABELS_PATH = INPUT_DIR / 'dependency_escalation_labels.csv'

# Check that all required files exist
for path in [HISTORY_PATH, ELIGIBLE_PATH, LABELS_PATH]:
    assert path.exists(), f'Missing file: {path}'
    print(f'Found: {path.name}')

print('\nAll three dependency evaluation files found!')

Found: final_dependency_history.csv
Found: final_eligible_dependency_adds.csv
Found: dependency_escalation_labels.csv

All three dependency evaluation files found!


## 2. Load data and validate schema

In [13]:
for name, df in [('history', history), ('eligible', eligible)]:
    parsed = pd.to_datetime(
        df['observed_at'],
        utc=True,
        format='mixed',
        errors='coerce'
    )

    print(f'\n{name.upper()}')
    print('Total rows:', len(df))
    print('Missing timestamps:', df['observed_at'].isna().sum())
    print('Unparseable timestamps:',
          (parsed.isna() & df['observed_at'].notna()).sum())


HISTORY
Total rows: 3379
Missing timestamps: 808
Unparseable timestamps: 0

ELIGIBLE
Total rows: 816
Missing timestamps: 0
Unparseable timestamps: 0


In [14]:
missing = history[history['observed_at'].isna()]

print("Missing timestamps by action:")
print(missing['action'].value_counts(dropna=False))

print("\nEligible events among missing timestamps:")
print(
    (
        missing['action'].eq('ADD')
        & missing['exclude_from_escalation'].eq(False)
    ).sum()
)

Missing timestamps by action:
action
MIGRATION    808
Name: count, dtype: int64

Eligible events among missing timestamps:
0


In [15]:
history=pd.read_csv(HISTORY_PATH,low_memory=False)
eligible=pd.read_csv(ELIGIBLE_PATH,low_memory=False)
labels=pd.read_csv(LABELS_PATH,low_memory=False)
required_events={'instance_id','event_uid','observed_at','action','upstream_issue','downstream_issue','exclude_from_escalation'}
required_labels={'instance_id','week_number','checkpoint_time','horizon_end','future_new_links','future_affected_downstream','dependency_risk_label'}
for name,df in [('history',history),('eligible',eligible)]:
    assert required_events.issubset(df.columns),(name,sorted(required_events-set(df.columns)))
    assert df['event_uid'].notna().all() and df['event_uid'].is_unique,name+' duplicate or null event_uid'
assert required_labels.issubset(labels.columns),sorted(required_labels-set(labels.columns))
assert not labels.duplicated(['instance_id','week_number','checkpoint_time']).any()
for name, df in [('history', history), ('eligible', eligible)]:
    df['event_time_utc'] = pd.to_datetime(
        df['observed_at'],
        utc=True,
        format='mixed',
        errors='coerce'
    )

    missing_original = df['observed_at'].isna()
    unparseable = df['event_time_utc'].isna() & ~missing_original

    # Existing timestamps must always be parseable.
    assert not unparseable.any(), (
        f'Unparseable non-missing timestamps in {name}'
    )

    if name == 'eligible':
        # Every eligible ADD event needs a timestamp.
        assert df['event_time_utc'].notna().all(), (
            'Eligible dependency events contain missing timestamps'
        )

    print(
        f'{name}: {len(df)} rows, '
        f'{int(missing_original.sum())} missing timestamps, '
        f'{int(unparseable.sum())} unparseable timestamps'
    )
labels['checkpoint_utc']=pd.to_datetime(labels['checkpoint_time'],utc=True,format='mixed',errors='coerce')
labels['horizon_utc']=pd.to_datetime(labels['horizon_end'],utc=True,format='mixed',errors='coerce')
assert labels[['checkpoint_utc','horizon_utc']].notna().all().all()
assert (labels['horizon_utc']-labels['checkpoint_utc']).eq(pd.Timedelta(days=14)).all(),'Non-14-day horizon'
print('History:',len(history),'Eligible:',len(eligible),'Labels:',len(labels))

history: 3379 rows, 808 missing timestamps, 0 unparseable timestamps
eligible: 816 rows, 0 missing timestamps, 0 unparseable timestamps
History: 3379 Eligible: 816 Labels: 987


## 3. Reconstruct eligible ADD events from processed history

In [16]:
def bool_series(s):
    v=s.astype('string').str.strip().str.lower()
    assert v.isin(['true','false']).all(), 'Unexpected exclusion flag value'
    return v.eq('true')
reconstructed=history.loc[history['action'].astype(str).str.upper().eq('ADD') & ~bool_series(history['exclude_from_escalation'])].copy()
keys=['instance_id','event_uid','action','upstream_issue','downstream_issue','event_time_utc']
comparison=reconstructed[keys].merge(eligible[keys],on='event_uid',how='outer',suffixes=('_reconstructed','_saved'),indicator=True,validate='one_to_one')
for field in ['instance_id','action','upstream_issue','downstream_issue','event_time_utc']:
    comparison[field+'_match']=comparison[field+'_reconstructed'].eq(comparison[field+'_saved']).fillna(False)
comparison['all_fields_match']=comparison['_merge'].eq('both') & comparison[[c for c in comparison if c.endswith('_match')]].all(axis=1)
errors=comparison.loc[~comparison['all_fields_match']].copy()
comparison.to_csv(OUT/'dependency_event_comparison.csv',index=False)
errors.to_csv(OUT/'dependency_event_discrepancies.csv',index=False)
print('Reconstructed eligible:',len(reconstructed),'Saved eligible:',len(eligible),'Discrepancies:',len(errors))
display(comparison['_merge'].value_counts())
if len(errors):display(errors.head(15))

Reconstructed eligible: 816 Saved eligible: 816 Discrepancies: 0


,count
_merge,
both,816
left_only,0
right_only,0


## 4. Recalculate checkpoint labels

For each checkpoint, count eligible ADD events in **(checkpoint, horizon_end]** and count distinct downstream issues. Positive label = at least 3 distinct downstream issues. The interval boundary follows the saved 14-day forward-window convention; if teammate code uses a different boundary, update it explicitly and document the change.

In [17]:
rows=[]
by_instance={k:g.sort_values('event_time_utc') for k,g in reconstructed.groupby('instance_id')}
for r in labels.itertuples(index=False):
    ev=by_instance.get(r.instance_id)
    if ev is None: new_links=affected=0
    else:
        window=ev.loc[(ev['event_time_utc']>r.checkpoint_utc)&(ev['event_time_utc']<=r.horizon_utc)]
        new_links=len(window)
        affected=window['downstream_issue'].nunique(dropna=True)
    rows.append({'instance_id':r.instance_id,'week_number':r.week_number,'checkpoint_time':r.checkpoint_time,
                 'saved_future_new_links':int(r.future_new_links),'reconstructed_future_new_links':int(new_links),
                 'saved_future_affected_downstream':int(r.future_affected_downstream),'reconstructed_future_affected_downstream':int(affected),
                 'saved_dependency_risk_label':int(r.dependency_risk_label),'reconstructed_dependency_risk_label':int(affected>=3)})
result=pd.DataFrame(rows)
for f in ['future_new_links','future_affected_downstream','dependency_risk_label']:
    result[f+'_match']=result['saved_'+f].eq(result['reconstructed_'+f])
result['all_match']=result[[c for c in result if c.endswith('_match')]].all(axis=1)
result.to_csv(OUT/'dependency_label_comparison.csv',index=False)
label_errors=result.loc[~result['all_match']]
label_errors.to_csv(OUT/'dependency_label_discrepancies.csv',index=False)
print('Matched all fields:',int(result['all_match'].sum()),'/',len(result))
display(result[[c for c in result if c.endswith('_match')]].sum())
if len(label_errors):display(label_errors.head(15))

Matched all fields: 987 / 987


,0
future_new_links_match,987
future_affected_downstream_match,987
dependency_risk_label_match,987
all_match,987


## 5. Reproducibility summary and input provenance

In [19]:
PATHS = {
    'history': HISTORY_PATH,
    'eligible': ELIGIBLE_PATH,
    'labels': LABELS_PATH
}

In [21]:
import hashlib

PATHS = {
    'history': HISTORY_PATH,
    'eligible': ELIGIBLE_PATH,
    'labels': LABELS_PATH
}

def sha256(path):
    hasher = hashlib.sha256()

    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            hasher.update(chunk)

    return hasher.hexdigest()

In [22]:
summary={
 'evaluation':'Phase2 Dependency Ground Truth Processed-Data Consistency',
 'history_records':int(len(history)),
 'history_actions':{str(k):int(v) for k,v in history['action'].value_counts().items()},
 'reconstructed_eligible_adds':int(len(reconstructed)),
 'saved_eligible_adds':int(len(eligible)),
 'event_comparison_discrepancies':int(len(errors)),
 'checkpoint_labels':int(len(result)),
 'future_new_links_matches':int(result['future_new_links_match'].sum()),
 'future_affected_downstream_matches':int(result['future_affected_downstream_match'].sum()),
 'dependency_risk_label_matches':int(result['dependency_risk_label_match'].sum()),
 'label_rows_with_any_discrepancy':int(len(label_errors)),
 'inputs':{k:{'filename':v.name,'sha256':sha256(v)} for k,v in PATHS.items()},
 'time_window':'(checkpoint_time, checkpoint_time + 14 days]',
 'threshold_distinct_downstream_issues':3,
 'independent_original_jira_verification':False,
 'limitations':['Eligibility exclusions and direction already assigned upstream in processed history.',
                'Exact raw Jira event classification and issue-to-instance membership are not independently verified.',
                'Reproduction of synthetic operational labels is not external risk construct validity.']}
(OUT/'dependency_validation_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))
print('Saved outputs to:',OUT)
assert len(errors)==0,'Event export discrepancies detected: inspect discrepancy CSV'
assert len(label_errors)==0,'Label discrepancies detected: inspect discrepancy CSV'
print('PASS: processed-history-to-label reproducibility checks')

{
  "evaluation": "Phase2 Dependency Ground Truth Processed-Data Consistency",
  "history_records": 3379,
  "history_actions": {
    "ADD": 1659,
    "REMOVE": 912,
    "MIGRATION": 808
  },
  "reconstructed_eligible_adds": 816,
  "saved_eligible_adds": 816,
  "event_comparison_discrepancies": 0,
  "checkpoint_labels": 987,
  "future_new_links_matches": 987,
  "future_affected_downstream_matches": 987,
  "dependency_risk_label_matches": 987,
  "label_rows_with_any_discrepancy": 0,
  "inputs": {
    "history": {
      "filename": "final_dependency_history.csv",
      "sha256": "1a59920752b735e2b013205227c55b358eb4168daeafc85e01a31fb0c698927c"
    },
    "eligible": {
      "filename": "final_eligible_dependency_adds.csv",
      "sha256": "f1e6577322ffc65563346fc17fe1fb0f1b5492f91ec1d7a913b4f9a5c03cf707"
    },
    "labels": {
      "filename": "dependency_escalation_labels.csv",
      "sha256": "d4db2905f275f3d2ba1fedfbcc4a15ac6a9a73c44afec7af03ec22eba62d02f2"
    }
  },
  "time_window"

## Interpretation

A clean run demonstrates exact agreement between the saved processed history, eligible-event export, and 14-day dependency labels. It **does not** prove that upstream event classification, exclusion flags, or the risk construct match independent original Jira ground truth. Preserve input hashes and discrepancy tables with the report.